# 🗺️ V43: THE TRUE TOPOLOGICAL CASCADE 🗺️\nEvolusi akhir dari arsitektur geospasial kita. V43 menggabungkan komunikasi Hulu-Hilir dinamis (dari V26.5) dengan Fitur Fisika Cuaca Ekstrim (dari V37). Dan yang paling mematikan: Rute sungai tidak lagi ditebak pakai jarak lurus, melainkan 100% menggunakan Silsilah Topologi Asli (`NEXT_DOWN`) dari database HydroRIVERS!

In [ ]:
# 1. MOUNT GOOGLE DRIVE & INSTALL GEOPANDAS
!pip install -q geopandas shapely optuna xgboost ipywidgets

from google.colab import drive
import os
import warnings
warnings.filterwarnings('ignore')

print("Meminta Izin Akses Google Drive...")
drive.mount('/content/drive')

# UBAH BARIS INI SESUAI DENGAN FOLDER ANDA
PROJECT_PATH = '/content/drive/MyDrive/SSF-SSDS/'
os.chdir(PROJECT_PATH)
print(f"Bekerja di direktori: {os.getcwd()}")

In [ ]:
# 2. IMPORT LIBRARY & LOAD DATA
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import json

print("Memuat Dataset Utama...")
train_df = pd.read_csv('train.csv', parse_dates=['datetime'])
test_df = pd.read_csv('test.csv')
env_df = pd.read_csv('data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
sample_sub = pd.read_csv('sample_submission.csv')
coords_df = pd.read_csv('data_pendukung/koordinat_pos.csv')

# Parsing Test Set
test_parsed = test_df.copy()
split_info = test_parsed['id'].str.split(' - ', n=1, expand=True)
test_parsed['datetime'] = pd.to_datetime(split_info[0])
test_parsed['nama_pos'] = split_info[1]

print("Data berhasil dimuat!")

### TAHAP 1: EKSTRAKSI TOPOLOGI ASLI HYDRORIVERS (NEXT_DOWN)

In [ ]:
print("1. Memuat Shapefile HydroRIVERS...")
rivers = gpd.read_file('data_pendukung/HydroRIVERS_v10_au_shp/HydroRIVERS_v10_au.shp')
rivers_bs = rivers.cx[110.5:113.0, -8.5:-6.5].copy() # Crop ke Bengawan Solo

gdf_coords = gpd.GeoDataFrame(
    coords_df, 
    geometry=gpd.points_from_xy(coords_df.longitude, coords_df.latitude), 
    crs="EPSG:4326"
)

print("2. Spatial Join: Mencocokkan Stasiun dengan Garis Sungai...")
station_hyriv = {}
for idx, row in gdf_coords.iterrows():
    distances = rivers_bs.geometry.distance(row.geometry)
    nearest_idx = distances.idxmin()
    hyriv_id = rivers_bs.loc[nearest_idx, 'HYRIV_ID']
    station_hyriv[row['nama_pos']] = hyriv_id

print("3. Melacak Silsilah Hulu-Hilir (NEXT_DOWN)...")
next_down_map = rivers_bs.set_index('HYRIV_ID')['NEXT_DOWN'].to_dict()
upstream_map = {pos: [] for pos in coords_df['nama_pos']}

for pos, start_hyriv in station_hyriv.items():
    current_hyriv = start_hyriv
    visited = set()
    while current_hyriv in next_down_map and current_hyriv != 0:
        visited.add(current_hyriv)
        current_hyriv = next_down_map[current_hyriv]
        for other_pos, other_hyriv in station_hyriv.items():
            if pos != other_pos and current_hyriv == other_hyriv:
                if pos not in upstream_map[other_pos]:
                    upstream_map[other_pos].append(pos)
                    print(f"   🌊 Aliran Ditemukan: {pos} (Hulu) --> {other_pos} (Hilir)")
        if current_hyriv in visited:
            break

print("Topologi Asli berhasil diekstrak!")

### TAHAP 2: OTOT FISIKA (EWM & Rolling Cuaca)

In [ ]:
print("Mengekstraksi Fitur Cuaca Lokal & EWM (V37)...")
env_df = env_df.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)
grouped = env_df.groupby('nama_pos')

agg_df = env_df[['datetime', 'nama_pos', 'rainfall_mm', 'temperature_c', 'soil_moisture_0_7cm']].copy()

if 'soil_moisture_28_100cm' in env_df.columns:
    agg_df['soil_moisture_28_100cm'] = env_df['soil_moisture_28_100cm']

# Fitur Cuaca Berat
agg_df['rain_sum_72h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(72, min_periods=1).sum())
agg_df['rain_sum_168h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(168, min_periods=1).sum())
agg_df['rain_ewm_48h'] = grouped['rainfall_mm'].transform(lambda x: x.ewm(halflife=48, ignore_na=True).mean())
agg_df['soil_mean_72h'] = grouped['soil_moisture_0_7cm'].transform(lambda x: x.rolling(72, min_periods=1).mean())
agg_df['temp_mean_72h'] = grouped['temperature_c'].transform(lambda x: x.rolling(72, min_periods=1).mean())

env_feat = agg_df.copy()
print("Fitur Cuaca siap!")

### TAHAP 3: MERAKIT KASKADE (Mentransfer Hujan Hulu ke Hilir)

In [ ]:
def prepare_dataset_cascade(base_df, env_feat, historical_tma_df, upstream_map):
    df = base_df.copy()
    
    # 1. Fitur Siklik (Sihir Penanggalan)
    df['hour'] = df['datetime'].dt.hour
    df['month'] = df['datetime'].dt.month
    df['dayofyear'] = df['datetime'].dt.dayofyear
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12.0)
    
    df = pd.merge(df, env_feat, on=['datetime', 'nama_pos'], how='left')
    
    # 2. Memori Stasiun Sendiri (Lokal)
    historical_map = historical_tma_df.set_index(['nama_pos', 'datetime'])['tma_mdpl'].to_dict()
    df['tma_lag_365d'] = df.apply(lambda row: historical_map.get((row['nama_pos'], row['datetime'] - pd.DateOffset(years=1)), np.nan), axis=1)
    
    last_known_map = historical_tma_df.dropna(subset=['tma_mdpl']).sort_values('datetime').groupby('nama_pos').tail(1).set_index('nama_pos')['tma_mdpl'].to_dict()
    df['last_known_tma'] = df['nama_pos'].map(last_known_map)
    
    # 3. KASKADE: TRANSFER DATA HULU
    print("Menyuntikkan Data Kiriman Hulu (Upstream Topology)...")
    upstream_dfs = []
    
    for pos, upstreams in upstream_map.items():
        if len(upstreams) > 0:
            up_df = df[df['nama_pos'].isin(upstreams)]
            agg = up_df.groupby('datetime')[['rain_sum_168h', 'rain_ewm_48h', 'last_known_tma']].mean().reset_index()
        else:
            up_df = df[df['nama_pos'] == pos]
            agg = up_df[['datetime', 'rain_sum_168h', 'rain_ewm_48h', 'last_known_tma']].copy()
        
        agg = agg.rename(columns={
            'rain_sum_168h': 'upstream_rain_sum_168h',
            'rain_ewm_48h': 'upstream_rain_ewm_48h',
            'last_known_tma': 'upstream_tma_mdpl'
        })
        agg['nama_pos'] = pos
        upstream_dfs.append(agg)
        
    all_upstream = pd.concat(upstream_dfs, ignore_index=True)
    df = pd.merge(df, all_upstream, on=['datetime', 'nama_pos'], how='left')
    
    return df

historical_tma = train_df[['datetime', 'nama_pos', 'tma_mdpl']].copy()

print("Memproses Train Set...")
train_full = prepare_dataset_cascade(train_df, env_feat, historical_tma, upstream_map)
print("Memproses Test Set...")
test_full = prepare_dataset_cascade(test_parsed, env_feat, historical_tma, upstream_map)

# Forward Fill Memori
for st in train_full['nama_pos'].unique():
    mask_tr = train_full['nama_pos'] == st
    train_full.loc[mask_tr, 'tma_lag_365d'] = train_full.loc[mask_tr, 'tma_lag_365d'].fillna(train_full.loc[mask_tr, 'last_known_tma'])
    mask_te = test_full['nama_pos'] == st
    test_full.loc[mask_te, 'tma_lag_365d'] = test_full.loc[mask_te, 'tma_lag_365d'].fillna(test_full.loc[mask_te, 'last_known_tma'])

train_full = train_full.dropna(subset=['tma_mdpl']).reset_index(drop=True)

drop_cols = ['datetime', 'nama_pos', 'tma_mdpl', 'id']
features = [c for c in train_full.columns if c not in drop_cols]
print(f"Total Fitur Kaskade V43: {len(features)}")

### TAHAP 4: OPTUNA TWEEDIE DENGAN TIME-SERIES SPLIT

In [ ]:
stations = train_full['nama_pos'].unique()
station_models = {}
station_scores = {}

optuna.logging.set_verbosity(optuna.logging.ERROR)
N_TRIALS = 20
N_SPLITS = 3

print(f"Memulai Training V43 untuk 30 Stasiun (Total {30 * N_TRIALS} Trials)...\n")

for st in stations:
    X_st = train_full[train_full['nama_pos'] == st][features]
    y_st = train_full[train_full['nama_pos'] == st]['tma_mdpl']
    
    def objective(trial):
        params = {
            'objective': 'reg:tweedie', 
            'tweedie_variance_power': trial.suggest_float('tweedie_variance_power', 1.1, 1.9),
            'eval_metric': 'rmse', 
            'tree_method': 'hist',
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'n_estimators': 600,
            'random_state': 42
        }
        
        tscv = TimeSeriesSplit(n_splits=N_SPLITS)
        scores = []
        for train_idx, val_idx in tscv.split(X_st):
            X_tr, y_tr = X_st.iloc[train_idx], np.clip(y_st.iloc[train_idx], 0.0001, None)
            X_va, y_va = X_st.iloc[val_idx], np.clip(y_st.iloc[val_idx], 0.0001, None)
            
            model = xgb.XGBRegressor(**params, early_stopping_rounds=30)
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            
            val_preds = model.predict(X_va)
            scores.append(np.sqrt(mean_squared_error(y_va, val_preds)))
            
        return np.mean(scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS)
    
    best_params = study.best_params
    best_params['objective'] = 'reg:tweedie'
    best_params['eval_metric'] = 'rmse'
    best_params['tree_method'] = 'hist'
    best_params['n_estimators'] = 600
    
    # Final Model Training
    models = []
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    for train_idx, val_idx in tscv.split(X_st):
        X_tr, y_tr = X_st.iloc[train_idx], np.clip(y_st.iloc[train_idx], 0.0001, None)
        X_va, y_va = X_st.iloc[val_idx], np.clip(y_st.iloc[val_idx], 0.0001, None)
        
        model = xgb.XGBRegressor(**best_params, early_stopping_rounds=30)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        models.append(model)
        
    station_models[st] = models
    station_scores[st] = study.best_value
    print(f"[{st}] - RMSE: {study.best_value:.4f}")

global_rmse = np.mean(list(station_scores.values()))
print(f"\n🏆 GRAND AVERAGE RMSE ACROSS 30 STATIONS: {global_rmse:.4f}")

### TAHAP 5: EVALUASI & SUBMISSION

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

stations_list = sorted(list(stations))

def plot_importance_and_pred(station_name):
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 2, figsize=(22, 7))
    
    # 1. Plot Feature Importance
    feat_imp = np.mean([m.feature_importances_ for m in station_models[station_name]], axis=0)
    imp_df = pd.DataFrame({'Features': features, 'Importance score': feat_imp})
    imp_df = imp_df.sort_values(by='Importance score', ascending=False).head(15)
    sns.barplot(x='Importance score', y='Features', data=imp_df, ax=ax[0], palette='magma')
    ax[0].set_title(f'Pembedahan Otak (Feature Importance) - [{station_name}]')
    
    # 2. Plot Actual vs Predicted (250 hari terakhir dari Train Set)
    X_target = train_full[train_full['nama_pos'] == station_name][features]
    y_target = train_full[train_full['nama_pos'] == station_name]['tma_mdpl']
    
    val_preds = np.zeros(len(X_target))
    for m in station_models[station_name]:
        val_preds += m.predict(X_target) / N_SPLITS
        
    sub_actual = y_target.iloc[-250:].values
    sub_pred = val_preds[-250:]
    
    ax[1].plot(sub_actual, label='Aktual TMA', color='blue', alpha=0.7, linewidth=2.5)
    ax[1].plot(sub_pred, label='Prediksi Kaskade (V43)', color='red', linestyle='--', alpha=0.8, linewidth=2)
    ax[1].set_title(f'Aktual vs Prediksi - 250 Hari Terakhir [{station_name}]')
    ax[1].legend()
    
    plt.tight_layout()
    plt.show()

widgets.interactive(plot_importance_and_pred, station_name=widgets.Dropdown(options=stations_list, description='Stasiun:'))

In [ ]:
print("Mencetak Submission File...")
test_preds = np.zeros(len(test_full))

for st in stations:
    idx = test_full['nama_pos'] == st
    if idx.sum() == 0: continue
        
    X_test_st = test_full.loc[idx, features]
    
    st_preds = np.zeros(len(X_test_st))
    for m in station_models[st]:
        st_preds += m.predict(X_test_st) / N_SPLITS
        
    test_full.loc[idx, 'tma_mdpl'] = st_preds

sub_pred = test_full[['id', 'tma_mdpl']].copy()
sub = pd.merge(sample_sub[['id']], sub_pred, on='id', how='left')
sub['tma_mdpl'] = sub['tma_mdpl'].fillna(method='ffill').fillna(method='bfill')

output_file = 'submission_V43_topological_cascade.csv'
sub.to_csv(output_file, index=False)
print(f"✅ Selesai! File '{output_file}' siap disubmit ke Kaggle!")